## Preprocess the dataset

In [1]:
# Import libraries
from datasets import load_dataset
from transformers import AutoProcessor, AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
import wave
import numpy as np
import pandas as pd
import os
import re
import string

/home/dhuser/.pyenv/versions/3.11.7/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Retrieve .txt files
def list_files_in_directory(directory):
    file_list = []
    for filename in os.listdir(directory):
        # Only pick up files with .txt extensions (transcript)
        if filename.endswith(".txt"):
            file_list.append(filename.replace(".txt", ""))
    return file_list

# Create the dataframe
def get_reference_df(directory, audio_txt_file):
    txt_file_path = os.path.join(directory, audio_txt_file + ".txt")
    columns = ["start_time", "end_time", "reference"]
    # Read the text file into a DataFrame
    df = pd.read_csv(txt_file_path, sep="\t", header=None, names=columns, quoting=3)

    # Add file name
    df.insert(0, 'file_name', pd.Series([audio_txt_file] * len(df)))

    # Remove quotation marks
    df['reference'] = df['reference'].apply(lambda x : x.replace('"',""))
    
    return df

# Trim audio files
def trim_wav_by_timestamps(directory, wav_file_name, reference_df):
    # Create the output directory if it doesn't exist
    output_dir = "data/sub/"
    os.makedirs(output_dir, exist_ok=True)
    wav_file = os.path.join(directory, wav_file_name + ".wav") # get into data file
    
    # Load the WAV file
    audio = AudioSegment.from_wav(wav_file)
    
    def trim_segments(row):
        start_ms = float(row['start_time']) * 1000  # Convert start time to milliseconds
        end_ms = float(row['end_time']) * 1000      # Convert end time to milliseconds
        trimmed_segment = audio[start_ms:end_ms]
    
        return trimmed_segment
    
    # Iterate over timestamps and trim the audio
    for i, row in reference_df.iterrows():
        trimmed_segment = trim_segments(row)
        output_file = os.path.join(output_dir, wav_file_name + "_" f"trimmed_segment_{i+1}.wav")
        trimmed_segment.export(output_file, format="wav")
        reference_df.at[i, 'trimmed_segment_path'] = output_file
    
    return reference_df

# Helper function to remove punctuations from original subtitles
def strip_punctuation(text):
    # Create a translation table that maps each punctuation character to None
    translator = str.maketrans('', '', string.punctuation)
    # Use the translation table to remove all punctuation from the text
    return text.translate(translator)

# Filter english text and append it to dataframe
def filter_subs_by_lang(reference_df):
    # Helper function that is applied across the rows to filter english text only
    
    def filter_english_only(text):
        # Define a regex pattern to match English words
        english_pattern = re.compile(r'\b[A-Za-z]+\b')
        # Find all English words in the text
        english_words = english_pattern.findall(text)
        # Join the English words into a single string
        english_text = ' '.join(english_words)
        return english_text
    
    def filter_thai_only(text):
        # Remove punctuation from text
        text = strip_punctuation(text)
        # Tokenize the string, split by spaces
        list_of_words = text.split()
        # Define a regex pattern to match English words
        english_pattern = re.compile(r'\b[A-Za-z]+\b')
        # Find all English words in the text
        english_words = english_pattern.findall(text)
        # Find all Thai words
        thai_words = [word for word in list_of_words if word not in english_words]
        # Concatenate the Thai words into a string
        thai_text = ' '.join(thai_words)
        # Return thai string
        return thai_text

    reference_df['eng_reference'] = reference_df['reference'].apply(filter_english_only)
    reference_df['thai_reference'] = reference_df['reference'].apply(filter_thai_only)

    return reference_df

def get_combined_audio_table(directory, file_names):
    combined_df = pd.DataFrame()
    for file_name in file_names:
        # Reads the transcript dataframe which has the start_time, end_time of each transcript
        reference_df = get_reference_df(directory, file_name)

        # Split subtitles by language
        reference_df = filter_subs_by_lang(reference_df)
        # Remove 'reference' column
        reference_df = reference_df.drop('reference', axis=1)

        # Uncomment to trim all the .wav file according to the subtitles start_time and end_time
        ## reference_df = trim_wav_by_timestamps(directory, file_name, reference_df)
        
        # Append the processed DataFrame to the combined DataFrame
        combined_df = pd.concat([combined_df, reference_df], ignore_index=True)

        # Comment out this section if not required
        combined_df = combined_df.drop(['file_name', 'start_time', 'end_time'], axis=1)
    
    return combined_df

directory = os.path.join(os.getcwd(), "data/train/")
file_names = list_files_in_directory(directory)
combined_df = get_combined_audio_table(directory, file_names)

pd.options.display.max_colwidth = 100
combined_df

,eng_reference,thai_reference
0,So today I will treat them to,เดี๋ยววันเนี่ยจะพาไปเลี้ยง
1,spicy Tom Yum hot pot,ต้มยำหม้อไฟแซ่บๆ
2,Hello everyone This is,สวัสดีครับทุกคน เดือนนี้เป็นเดือน
3,a month of giving out bonuses,เมษายนเป็นเดือนแห่งการแจกโบนัสนะครับ
4,And our company is,และบริษัทของเราเนี่ยเป็น
...,...,...
2408,Please like share and subscribe,ดูแล้วชอบ กดไลก์ แชร์ ซับสไครบ์ด้วย
2409,Chollada Channel Chollada Channel,กับ ค่ะ
2410,Bye,บาย
2411,Please like share and subscribe,อย่าลืมกดไลก์ กดแชร์ หรือว่าซับสไครบ์


## Import SEALION Model

In [6]:
generation_kwargs = {
    "do_sample": False,  # set to true if temperature is not 0
    "temperature": None,
    "max_new_tokens": 256,
    "top_k": 50,
    "top_p": 0.7,
    "repetition_penalty": 1.2,
}

In [7]:
MODEL_NAME = "aisingapore/sea-lion-7b-instruct"

# Import tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "aisingapore/sea-lion-7b-instruct", 
    trust_remote_code=True
)

generation_kwargs["eos_token_id"] = tokenizer.eos_token_id

In [8]:
# Remove unneeded kwargs
if generation_kwargs["do_sample"] == False:
    generation_kwargs.pop("temperature")
    generation_kwargs.pop("top_k")
    generation_kwargs.pop("top_p")

In [9]:
# Import model
model = AutoModelForCausalLM.from_pretrained(
    "aisingapore/sea-lion-7b-instruct",
    trust_remote_code=True,
    device_map = 'auto'
)

The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.
Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.61s/it]


In [17]:
# Example Use Case

# Prompt template
prompt_template = "### USER:\nTranslate the following to English. Do not need to form full sentence. {human_prompt}\n\n### RESPONSE:\n"
prompt = """เมษายนเป็นเดือนแห่งการแจกโบนัสนะครับ"""
full_prompt = prompt_template.format(human_prompt=prompt)

tokens = tokenizer(full_prompt, return_tensors="pt")

output = model.generate(
    tokens["input_ids"],
    attention_mask = tokens["attention_mask"],
    **generation_kwargs,
)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


/home/dhuser/.cache/huggingface/modules/transformers_modules/aisingapore/sea-lion-7b-instruct/566afff3d12a72de5e2a0a627d233ad940fe7dcb/attention.py:124: UserWarning: Propagating key_padding_mask to the attention module and applying it within the attention module can cause unnecessary computation/memory usage. Consider integrating into attn_bias once and passing that to each attention module instead.
  warnings.warn(


### USER:
Translate the following to English. Do not need to form full sentence. เมษายนเป็นเดือนแห่งการแจกโบนัสนะครับ

### RESPONSE:
It is a month of bonus distribution in April.


In [18]:
# Test
test_df = combined_df.head(3)

# Prompt template
prompt_template = "### USER:\nTranslate the following to English. {human_prompt}\n\n### RESPONSE:\n"

def generate_translation(reference_df):

    def translate_thai(prompt):
        full_prompt = prompt_template.format(human_prompt=prompt)
        # Retrieve tokens
        tokens = tokenizer(full_prompt, return_tensors="pt")
        # Generate output
        output = model.generate(
            tokens["input_ids"],
            attention_mask = tokens["attention_mask"],
            **generation_kwargs,
        )
        # Decode output embeddings
        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
        print(decoded_output)
        # Trim to obtain output from formatted response
        translation = decoded_output.partition("RESPONSE:\n")[2]
        
        return translation

    reference_df['translation'] = reference_df['thai_reference'].apply(translate_thai)
    return reference_df

test_df = generate_translation(test_df)
test_df

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


### USER:
Translate the following to English. เดี๋ยววันเนี่ยจะพาไปเลี้ยง

### RESPONSE:
Let's go out for dinner tonight.


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


### USER:
Translate the following to English. ต้มยำหม้อไฟแซ่บๆ

### RESPONSE:
Let's make some hot and spicy soup!
### USER:
Translate the following to English. สวัสดีครับทุกคน เดือนนี้เป็นเดือน

### RESPONSE:
Hello everyone, it is June


/tmp/ipykernel_404430/1908051502.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reference_df['translation'] = reference_df['thai_reference'].apply(translate_thai)


,eng_reference,thai_reference,translation
0,So today I will treat them to,เดี๋ยววันเนี่ยจะพาไปเลี้ยง,Let's go out for dinner tonight.
1,spicy Tom Yum hot pot,ต้มยำหม้อไฟแซ่บๆ,Let's make some hot and spicy soup!
2,Hello everyone This is,สวัสดีครับทุกคน เดือนนี้เป็นเดือน,"Hello everyone, it is June"


In [16]:
# Prompt template
prompt_template = "### USER:\nTranslate the following to English. {human_prompt}\n\n### RESPONSE:\n"

# Generate tokens and generate output
def generate_translation(reference_df):

    def translate_thai(prompt):
        full_prompt = prompt_template.format(human_prompt=prompt)
        # Retrieve tokens
        tokens = tokenizer(full_prompt, return_tensors="pt")
        # Generate output
        output = model.generate(
            tokens["input_ids"],
            attention_mask = tokens["attention_mask"],
            **generation_kwargs,
        )
        # Decode output embeddings
        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
        # Trim to obtain output from formatted response
        translation = decoded_output.partition("RESPONSE:\n")[2]

        return translation

    reference_df['translation'] = reference_df['thai_reference'].apply(translate_thai)
    return reference_df

combined_df = generate_translation(combined_df)
combined_df

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
/home/dhuser/.pyenv/versions/3.11.7/lib/python3.11/site-packages/transformers/generation/utils.py:1659: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
/home/dhuser/.cache/huggingface/modules/transformers_modules/aisingapore/sea-lion-7b-instruct/566afff3d12a72de5e2a0a627d233ad940fe7dcb/attention.py:124: UserWarning: Propagating key_padding_mask to the attention module and applying it within the attention module can cause unnecessary computation/memory usage. Consider integrating into attn_bias once and passing that to each attention module instead.
  warnings.warn(


,eng_reference,thai_reference,translation
0,So today I will treat them to,เดี๋ยววันเนี่ยจะพาไปเลี้ยง,Let's go out for dinner tonight.
1,spicy Tom Yum hot pot,ต้มยำหม้อไฟแซ่บๆ,Let's make some hot and spicy soup!
2,Hello everyone This is,สวัสดีครับทุกคน เดือนนี้เป็นเดือน,"Hello everyone, it is June"
3,a month of giving out bonuses,เมษายนเป็นเดือนแห่งการแจกโบนัสนะครับ,"April is a month of bonus distribution, right?"
4,And our company is,และบริษัทของเราเนี่ยเป็น,And our company is
...,...,...,...
2408,Please like share and subscribe,ดูแล้วชอบ กดไลก์ แชร์ ซับสไครบ์ด้วย,"I like this video, please click Like and Share with your friends!"
2409,Chollada Channel Chollada Channel,กับ ค่ะ,ใช้คำว่า with แทนได้ไหมคะ?
2410,Bye,บาย,Good evening
2411,Please like share and subscribe,อย่าลืมกดไลก์ กดแชร์ หรือว่าซับสไครบ์,"If you are a student, please do not use this service for academic purposes such as submitting as..."


## Inspect SEA-LION Model

In [48]:
# Import LLM
tokenizer = AutoTokenizer.from_pretrained(
    "aisingapore/sea-lion-7b-instruct", 
    trust_remote_code=True
)

prompt_template = "### USER:\nTranslate the following to English. {human_prompt}\n\n### RESPONSE:\n"
prompt = "ฉันหิว"
full_prompt = prompt_template.format(human_prompt=prompt)

# Preprocess the input data (Convert audio to tensors)
tokens = tokenizer(full_prompt, return_tensors="pt")
print(tokens)

{'input_ids': tensor([[ 71721,  66561, 249853,      4,  88172,    311,   2057,    332,   3839,
         249835,  66012, 226626,      4,      4,  29864, 229553, 249853,      4]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [49]:
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print(input_ids)

attention_mask = tokens["attention_mask"]
print(attention_mask)

[0, 0, 0]
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [50]:
# Input must be a tensor
embeddings = model.transformer.wte(tokens['input_ids'])
embeddings

tensor([[[ 0.0894,  0.1318, -0.2344,  ..., -0.2734,  0.0066,  0.0645],
         [-0.1807, -0.1040, -0.0310,  ..., -0.1475,  0.0605, -0.1748],
         [ 0.0771,  0.2490,  0.0620,  ..., -0.1924,  0.2793,  0.2256],
         ...,
         [-0.0471,  0.0400,  0.1934,  ..., -0.2070, -0.0137, -0.2119],
         [ 0.0771,  0.2490,  0.0620,  ..., -0.1924,  0.2793,  0.2256],
         [ 0.3184,  0.4160, -0.2832,  ..., -0.3652, -0.0737,  0.1523]]],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)

In [51]:
# Example Use Case

output = model.generate(
    inputs_embeds = embeddings,
    **generation_kwargs,
)
print(tokenizer.decode(output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


I am hungry.
